# Quick Test Suite: Validate Setup

Run this notebook to quickly verify everything is working before executing the full density analysis.

**Time to complete**: 10-15 minutes

---

## Fix: OpenMP Conflict Resolution

If you see 'libiomp5md.dll already initialized' errors, run this cell first.

In [1]:
import os

# Fix for: OMP: Error #15: libiomp5md.dll already initialized
# This occurs when multiple packages link Intel OpenMP runtime
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

print("OpenMP conflict mitigation: ENABLED")
print("Set KMP_DUPLICATE_LIB_OK=True")
print()
print("Note: This is a temporary workaround.")
print("For a permanent fix, see the TROUBLESHOOTING.md file.")

OpenMP conflict mitigation: ENABLED
Set KMP_DUPLICATE_LIB_OK=True

Note: This is a temporary workaround.
For a permanent fix, see the TROUBLESHOOTING.md file.


## Test 1: Python Dependencies

In [2]:
import sys

dependencies = {
    'torch': 'PyTorch (Deep Learning)',
    'cv2': 'OpenCV (Video Processing)',
    'yaml': 'PyYAML (Config Files)',
    'ultralytics': 'Ultralytics (YOLO)',
    'numpy': 'NumPy (Numerical Computing)',
    'psutil': 'psutil (System Monitoring)',
}

print("\n" + "="*60)
print("TEST 1: PYTHON DEPENDENCIES")
print("="*60)

missing = []
for module, name in dependencies.items():
    try:
        __import__(module)
        print(f"[OK] {name:.<50}")
    except ImportError:
        print(f"[NO] {name:.<50}")
        missing.append(module)

print("="*60)

if missing:
    print(f"\n[MISSING] {len(missing)} package(s) not installed:")
    print(f"  pip install {' '.join(missing)}")
    print(f"\nOr install all requirements:")
    print(f"  pip install -r ../../requirements.txt")
else:
    print("\n[OK] All dependencies installed!")

print()


TEST 1: PYTHON DEPENDENCIES
[OK] PyTorch (Deep Learning)...........................
[OK] OpenCV (Video Processing).........................
[OK] PyYAML (Config Files).............................
[OK] Ultralytics (YOLO)................................
[OK] NumPy (Numerical Computing).......................
[OK] psutil (System Monitoring)........................

[OK] All dependencies installed!



## Test 2: GPU/CUDA Availability

In [3]:
import torch

print("\n" + "="*60)
print("TEST 2: GPU/CUDA AVAILABILITY")
print("="*60)

print(f"CUDA Available:              {torch.cuda.is_available()}")
print(f"PyTorch Version:             {torch.__version__}")
print(f"CUDA Version:                {torch.version.cuda}")
print(f"Number of GPUs:              {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print()
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        props = torch.cuda.get_device_properties(i)
        free_bytes, total_bytes = torch.cuda.mem_get_info(i)
        print(f"  VRAM:                    {total_bytes / 1024**3:.2f} GB total, {free_bytes / 1024**3:.2f} GB free")
        print(f"  Compute Capability:     {props.major}.{props.minor}")
        print()
    print("✓ GPU ready for inference!")
else:
    print("\n⚠ No GPU detected. CPU inference will be significantly slower.")
    print("   Check: nvidia-smi")
    print("   Reinstall: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124")

print("="*60)
print()


TEST 2: GPU/CUDA AVAILABILITY
CUDA Available:              True
PyTorch Version:             2.6.0+cu124
CUDA Version:                12.4
Number of GPUs:              1

GPU 0: NVIDIA GeForce RTX 4090
  VRAM:                    23.99 GB total, 22.46 GB free
  Compute Capability:     8.9

✓ GPU ready for inference!



## Test 3: File Structure

In [4]:
from pathlib import Path

base_dir = Path(r"C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\test-setup-03-run-density")

print("\n" + "="*60)
print("TEST 3: FILE STRUCTURE")
print("="*60)
print(f"Base Directory: {base_dir}")
print()

required_files = {
    'data.yaml': 'Configuration file',
    'test-setup-03-run-density-yolo26-pose-count.ipynb': 'Main notebook',
    'input/20260329_34.mp4': 'Input video',
    '.yolo': 'YOLO cache directory',
    'output': 'Output directory',
}

all_exist = True
for file_path, description in required_files.items():
    full_path = base_dir / file_path
    exists = full_path.exists()
    status = "✓" if exists else "✗"
    print(f"{status} {file_path:.<45} {description}")
    if not exists:
        all_exist = False

print("="*60)

if all_exist:
    print("\n✓ File structure complete!")
else:
    print("\n✗ Some files missing. Run: setup-density-hardware-analysis.ipynb")

print()


TEST 3: FILE STRUCTURE
Base Directory: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\test-setup-03-run-density

✗ data.yaml.................................... Configuration file
✗ test-setup-03-run-density-yolo26-pose-count.ipynb Main notebook
✗ input/20260329_34.mp4........................ Input video
✗ .yolo........................................ YOLO cache directory
✗ output....................................... Output directory

✗ Some files missing. Run: setup-density-hardware-analysis.ipynb



## Test 4: Configuration Validity

In [ ]:
import yaml
import json
from pathlib import Path

config_path = Path(r"C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\test-setup-03-run-density\data.yaml")

print("\n" + "="*60)
print("TEST 4: CONFIGURATION VALIDITY")
print("="*60)

if not config_path.exists():
    print(f"✗ Config not found: {config_path}")
else:
    try:
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        
        print(f"✓ Config loaded successfully\n")
        
        # Validate key parameters
        checks = [
            ('Batch size (8-64)', 8 <= config['runtime']['batch_size'] <= 64),
            ('Image size (640-1280)', 640 <= config['inference']['imgsz'] <= 1280),
            ('Video stride (≥1)', config['inference']['vid_stride'] >= 1),
            ('Confidence (0-1)', 0 < config['inference']['conf'] < 1),
            ('IOU threshold (0-1)', 0 < config['inference']['iou'] < 1),
            ('Half precision set', 'half' in config['inference']),
            ('Allow TF32 set', 'allow_tf32' in config['runtime']),
        ]
        
        all_valid = True
        for check_name, passed in checks:
            status = "✓" if passed else "✗"
            print(f"{status} {check_name}")
            if not passed:
                all_valid = False
        
        print()
        print("Configuration Summary:")
        print(f"  Batch Size:     {config['runtime']['batch_size']}")
        print(f"  Image Size:     {config['inference']['imgsz']}")
        print(f"  Video Stride:   {config['inference']['vid_stride']}")
        print(f"  Half Precision: {config['inference']['half']}")
        print(f"  TF32 Mode:      {config['runtime']['allow_tf32']}")
        
        if all_valid:
            print("\n✓ Configuration is valid!")
        else:
            print("\n✗ Some configuration values are out of range.")
    
    except Exception as e:
        print(f"✗ Error reading config: {e}")

print("="*60)
print()

## Test 5: Video File Validation

In [ ]:
import cv2
from pathlib import Path

video_path = Path(r"C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\test-setup-03-run-density\input\20260329_34.mp4")

print("\n" + "="*60)
print("TEST 5: VIDEO FILE VALIDATION")
print("="*60)

if not video_path.exists():
    print(f"✗ Video not found: {video_path}")
else:
    print(f"✓ Video file exists")
    print(f"  Path: {video_path}")
    print(f"  Size: {video_path.stat().st_size / 1024**2:.2f} MB")
    print()
    
    cap = cv2.VideoCapture(str(video_path))
    
    if not cap.isOpened():
        print("✗ Could not open video with OpenCV")
    else:
        print("✓ Video opened successfully\n")
        
        # Get metadata
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"Video Metadata:")
        print(f"  Resolution:     {width}x{height}")
        print(f"  FPS:            {fps}")
        print(f"  Total Frames:   {frame_count}")
        print(f"  Duration:       {frame_count/fps:.1f} seconds")
        
        # Try reading first frame
        ret, frame = cap.read()
        if ret:
            print(f"\n✓ First frame read successfully")
            print(f"  Frame shape: {frame.shape}")
        else:
            print(f"\n✗ Could not read first frame")
        
        cap.release()

print("="*60)
print()

## Test 6: YOLO Model Loading

In [ ]:
import torch
from ultralytics import YOLO

print("\n" + "="*60)
print("TEST 6: YOLO MODEL LOADING")
print("="*60)
print("Loading YOLO26-X Pose model...\n")

try:
    model = YOLO('yolo26x-pose.pt', task='pose')
    print("✓ Model loaded successfully")
    
    # Get model info
    params = sum(p.numel() for p in model.model.parameters()) / 1e6
    print(f"  Model size:     {params:.1f}M parameters")
    
    # Move to GPU if available
    if torch.cuda.is_available():
        model.to('cuda:0')
        print(f"✓ Model moved to GPU")
    else:
        print(f"⚠ Using CPU (inference will be slow)")
    
    print()
    print("✓ YOLO26 model ready for inference!")

except Exception as e:
    print(f"✗ Error loading model: {e}")
    print("\nTroubleshooting:")
    print("  Run: pip install ultralytics -U")
    print("  Then: python -c 'from ultralytics import YOLO; YOLO(\"yolo26x-pose.pt\")'")

print("="*60)
print()

## Test 7: Final Validation Summary

In [ ]:
print("\n" + "="*60)
print("FINAL VALIDATION SUMMARY")
print("="*60)

all_pass = True

try:
    import torch
    print("✓ Dependencies installed")
except:
    print("✗ Dependencies missing")
    all_pass = False

if torch.cuda.is_available():
    print(f"✓ GPU available ({torch.cuda.get_device_name(0)})")
else:
    print("⚠ GPU not available (will use CPU)")

from pathlib import Path
base = Path(r"C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\test-setup-03-run-density")

if (base / 'data.yaml').exists():
    print("✓ Configuration file ready")
else:
    print("✗ Configuration file missing")
    all_pass = False

if (base / 'input' / '20260329_34.mp4').exists():
    print("✓ Input video available")
else:
    print("✗ Input video missing")
    all_pass = False

print("="*60)

if all_pass:
    print("\n✓ ALL TESTS PASSED!\n")
    print("You can now run the density analysis:")
    print(f"  {base / 'test-setup-03-run-density-yolo26-pose-count.ipynb'}")
else:
    print("\n✗ Some tests failed.")
    print("Please review errors above and fix before proceeding.")

print()